In [25]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from numpy import random
import gerrychain   
from gerrychain import Graph, Partition, proposals, updaters, constraints, accept, MarkovChain
from gerrychain.updaters import cut_edges, Tally
from gerrychain.tree import bipartition_tree
from gerrychain.proposals import recom
from gerrychain.accept import always_accept
from gerrychain.constraints import within_percent_of_ideal_population
from functools import partial

In [26]:
## making graph
sc_graph = Graph.from_file("./SC/SC.shp")
gdf = gpd.read_file("./SC/SC.shp")
sc_from_gpd_graph = Graph.from_geodataframe(gdf, adjacency="rook") ## creating graph from geodataframe

In [27]:
sc_graph.nodes[1]

{'boundary_node': False,
 'area': 8648536.552029321,
 'COUNTY': '013',
 'PCODE': '111',
 'CODE_NAME': 'Burton 1B',
 'G20PRER': 160,
 'G20PRED': 650,
 'G20USSR': 142,
 'G20USSD': 679,
 'G20USSCBLE': 7,
 'TOTPOP': 2305,
 'HISP': 383,
 'NH_WHITE': 591,
 'NH_BLACK': 1203,
 'NH_AMIN': 5,
 'NH_ASIAN': 25,
 'NH_NHPI': 1,
 'NH_OTHER': 19,
 'NH_2MORE': 78,
 'VAP': 1623,
 'HVAP': 1534,
 'WVAP': 5,
 'BVAP': 22,
 'AMINVAP': 1,
 'ASIANVAP': 156,
 'NHPIVAP': 89,
 'OTHERVAP': 84,
 '2MOREVAP': 13,
 'G18GOVD': 25.746192893401012,
 'G18GOVR': 0.893401015228426,
 'G18SOSD': 26.233502538071065,
 'G18SOSR': 0.649746192893401,
 'G18TRED': 25.82741116751269,
 'G18TRER': 0.649746192893401,
 'G18ATGD': 25.746192893401012,
 'G18ATGR': 0.568527918781726,
 'G18COMR': 3.32994923857868,
 'G18SPIR': 1.055837563451776,
 'G18AGRR': 1.299492385786802,
 'SEND': '45',
 'geometry': <POLYGON ((523042.065 3587876.474, 523009.381 3587902.893, 522993.181 358791...>}

In [28]:
gdf.columns

Index(['COUNTY', 'PCODE', 'CODE_NAME', 'G20PRER', 'G20PRED', 'G20USSR',
       'G20USSD', 'G20USSCBLE', 'TOTPOP', 'HISP', 'NH_WHITE', 'NH_BLACK',
       'NH_AMIN', 'NH_ASIAN', 'NH_NHPI', 'NH_OTHER', 'NH_2MORE', 'VAP', 'HVAP',
       'WVAP', 'BVAP', 'AMINVAP', 'ASIANVAP', 'NHPIVAP', 'OTHERVAP',
       '2MOREVAP', 'G18GOVD', 'G18GOVR', 'G18SOSD', 'G18SOSR', 'G18TRED',
       'G18TRER', 'G18ATGD', 'G18ATGR', 'G18COMR', 'G18SPIR', 'G18AGRR',
       'SEND', 'geometry'],
      dtype='object')

In [29]:
print(gdf["SEND"].unique())

['45' '43' '46' '24' '25' '40' '36' '26' '37' '44' '32' '41' '20' '38'
 '42' '14' '17' '27' '29' '30' '34' '9' '8' '7' '13' '12' '6' '2' '5' '28'
 '33' '35' '10' '23' '18' '39' '21' '22' '19' '15' '16' '31' '3' '4' '1'
 '11']


In [34]:
updaters_gov = {
    "population": Tally("TOTPOP", alias="population"),
    "cut_edges": cut_edges,
    "dem_votes": Tally("G18GOVD", alias="dem_votes"),
    "rep_votes": Tally("G18GOVR", alias="rep_votes")
}

updaters_atg = {
    "population": Tally("TOTPOP", alias="population"),
    "cut_edges": cut_edges,
    "dem_votes": Tally("G20ATGD", alias="dem_votes"),
    "rep_votes": Tally("G20ATGR", alias="rep_votes")
}

In [36]:
initial_partition_gov = Partition(
    graph=sc_from_gpd_graph,
    assignment="SEND",
    updaters=updaters_gov
)

In [37]:
ideal_pop = sum(initial_partition_gov["population"].values()) / len(set(initial_partition_gov.assignment.values()))
proposal = partial(
    recom,
    pop_col="TOTPOP",
    pop_target=ideal_pop,
    epsilon=0.02,
    node_repeats=1
)

chain = MarkovChain(
    proposal=proposal,
    constraints=[within_percent_of_ideal_population(initial_partition_gov, 0.10)], ##Tolerance +- 2%
    accept=always_accept,
    initial_state=initial_partition_gov,
    total_steps=1000
)

In [38]:
dem_wins = []
cut_edges_list = []

for i, partition in enumerate(chain):
    dem = sum(1 for d, r in zip(partition["dem_votes"].values(), partition["rep_votes"].values()) if d > r)
    dem_wins.append(dem)
    cut_edges_list.append(len(partition["cut_edges"]))


c:\Anaconda\envs\gerry\Lib\site-packages\gerrychain\tree.py:704: BipartitionWarning: 
Failed to find a balanced cut after 1000 attempts.
If possible, consider enabling pair reselection within your
MarkovChain proposal method to allow the algorithm to select
a different pair of districts for recombination.
  warnings.warn(


KeyboardInterrupt: 